# Gold Taxi Trips Benchmark

Each design executes the complete landing → bronze → silver → gold pipeline. The final integrated Taxi Trips Delta table is partitioned either by `date` or by `pickup_borough`.

In [11]:
import json
import sys
from datetime import datetime, timezone
from importlib import reload
from statistics import median
from time import perf_counter

from pyspark.sql import SparkSession, functions as F

BENCHMARK_ROOT = "s3a://delta-data/benchmark/"
DESIGNS = {
    "by_date": "date",
    "by_pickup_borough": "pickup_borough",
}
CONFIG_PATHS = [
    "/opt/spark/configs/taxi_zones.json",
    "/opt/spark/configs/taxi_trips.json",
    "/opt/spark/configs/air_quality.json",
    "/opt/spark/configs/weather.json",
]

In [ ]:
def create_spark():
    spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName("Gold-Taxi-Benchmark")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    return spark

spark = create_spark()

In [13]:
JOB_PATH = "/opt/spark/jobs"
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

import bronze_job as bronze_module
import silver_job as silver_module
import gold_job as gold_module

reload(bronze_module)
reload(silver_module)
reload(gold_module)

from bronze_job import bronze_job
from gold_job import gold_job
from helpers import load_config
from metadata import create_run_context
from silver_job import silver_job

end_to_end_run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def benchmark_locations(design):
    root = f"{BENCHMARK_ROOT}gold_taxi/{end_to_end_run_id}/{design}"
    return {
        "bronze": f"{root}/bronze/bronze_",
        "silver": f"{root}/silver/silver_",
        "gold": f"{root}/gold/gold_",
    }

def run_end_to_end_benchmark(design):
    locations = benchmark_locations(design)
    run_context = create_run_context()
    started_at = perf_counter()

    configs = [load_config(path) for path in CONFIG_PATHS]
    for config in configs:
        bronze_job(spark, config, run_context, locations["bronze"])

    for config in configs:
        silver_job(
            spark, config, run_context,
            locations["bronze"], locations["silver"],
            write_metadata=False,
        )

    gold_job(
        spark,
        "integrated_taxi_trips",
        DESIGNS[design],
        locations["gold"],
        locations["silver"],
    )

    return {
        "design": design,
        "partition_column": DESIGNS[design],
        "ingestion_seconds": round(perf_counter() - started_at, 3),
        "gold_target": f"{locations['gold']}integrated_taxi_trips",
    }

In [14]:
date_result = run_end_to_end_benchmark("by_date")
print(json.dumps(date_result, indent=2))

Source files:
  file:///opt/spark/landing/taxi_zone_lookup/taxi_zone_lookup.csv
Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/bronze/bronze_taxi_zones
Source files:
  file:///opt/spark/landing/taxi/yellow_tripdata_2024-01.parquet
  file:///opt/spark/landing/taxi/yellow_tripdata_2024-02.parquet
  file:///opt/spark/landing/taxi/yellow_tripdata_2024-03.parquet


Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/bronze/bronze_taxi_trips
Source files:
  file:///opt/spark/landing/air_quality/hourly_88101_2024.csv


Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/bronze/bronze_air_quality
Source files:
  file:///opt/spark/landing/weather/weather.csv
Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/bronze/bronze_weather
Schema is valid.
[WARN]: Validation failed: column 'Borough' contains null or NaN values. Invalid rows: 1 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'Zone' contains null or NaN values. Invalid rows: 1 (since action='warn' was specified, ignore this)
Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/silver/silver_taxi_zones
Schema is valid.
Deduplicating records using columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amou

[WARN]: Validation failed: column 'passenger_count' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'RatecodeID' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'store_and_fwd_flag' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'congestion_surcharge' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'Airport_fee' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
Validation failed: column 'fare_amount' contains negative values. Removing 136567 invalid rows.
Validation failed: column 'extra' contains negative values. Removing 57864 invalid rows.
Validation failed: column 'mta_tax' contains negative values.

Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/silver/silver_taxi_trips
Schema is valid.
Deduplicating records using columns: ['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Date Local', 'Time Local', 'Date GMT', 'Time GMT', 'Sample Measurement', 'Units of Measure', 'MDL', 'Uncertainty', 'Qualifier', 'Method Type', 'Method Code', 'Method Name', 'State Name', 'County Name', 'Date of Last Change'].


Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/silver/silver_air_quality
Schema is valid.
Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/silver/silver_weather
joining taxi data with zones
joining taxi data with weather
joining taxi data with air quality
joining taxi data with air quality
writing gold data


Written gold Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/gold/gold_integrated_taxi_trips
{
  "design": "by_date",
  "partition_column": "date",
  "ingestion_seconds": 93.827,
  "gold_target": "s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/gold/gold_integrated_taxi_trips"
}


In [15]:
borough_result = run_end_to_end_benchmark("by_pickup_borough")
print(json.dumps(borough_result, indent=2))

Source files:
  file:///opt/spark/landing/taxi_zone_lookup/taxi_zone_lookup.csv
Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/bronze/bronze_taxi_zones
Source files:
  file:///opt/spark/landing/taxi/yellow_tripdata_2024-01.parquet
  file:///opt/spark/landing/taxi/yellow_tripdata_2024-02.parquet
  file:///opt/spark/landing/taxi/yellow_tripdata_2024-03.parquet


Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/bronze/bronze_taxi_trips
Source files:
  file:///opt/spark/landing/air_quality/hourly_88101_2024.csv


Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/bronze/bronze_air_quality
Source files:
  file:///opt/spark/landing/weather/weather.csv
Written bronze Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/bronze/bronze_weather
Schema is valid.
[WARN]: Validation failed: column 'Borough' contains null or NaN values. Invalid rows: 1 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'Zone' contains null or NaN values. Invalid rows: 1 (since action='warn' was specified, ignore this)
Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/silver/silver_taxi_zones
Schema is valid.
Deduplicating records using columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_t

[WARN]: Validation failed: column 'passenger_count' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'RatecodeID' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'store_and_fwd_flag' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'congestion_surcharge' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
[WARN]: Validation failed: column 'Airport_fee' contains null or NaN values. Invalid rows: 751962 (since action='warn' was specified, ignore this)
Validation failed: column 'fare_amount' contains negative values. Removing 136567 invalid rows.
Validation failed: column 'extra' contains negative values. Removing 57864 invalid rows.
Validation failed: column 'mta_tax' contains negative values.

Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/silver/silver_taxi_trips
Schema is valid.
Deduplicating records using columns: ['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Date Local', 'Time Local', 'Date GMT', 'Time GMT', 'Sample Measurement', 'Units of Measure', 'MDL', 'Uncertainty', 'Qualifier', 'Method Type', 'Method Code', 'Method Name', 'State Name', 'County Name', 'Date of Last Change'].


Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/silver/silver_air_quality
Schema is valid.
Written silver Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/silver/silver_weather
joining taxi data with zones
joining taxi data with weather
joining taxi data with air quality
joining taxi data with air quality
writing gold data


Written gold Delta table to: s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/gold/gold_integrated_taxi_trips
{
  "design": "by_pickup_borough",
  "partition_column": "pickup_borough",
  "ingestion_seconds": 113.293,
  "gold_target": "s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/gold/gold_integrated_taxi_trips"
}


In [16]:
def get_storage_metrics(target):
    jvm = spark.sparkContext._jvm
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    path = jvm.org.apache.hadoop.fs.Path(target)
    filesystem = path.getFileSystem(hadoop_conf)
    files = filesystem.listFiles(path, True)

    data_files = data_bytes = all_files = all_bytes = 0
    while files.hasNext():
        status = files.next()
        if status.isFile():
            all_files += 1
            all_bytes += status.getLen()
            if str(status.getPath().getName()).endswith(".parquet"):
                data_files += 1
                data_bytes += status.getLen()

    return {
        "data_files": data_files,
        "total_files": all_files,
        "data_size_mib": round(data_bytes / 1024**2, 2),
        "total_size_mib": round(all_bytes / 1024**2, 2),
    }

benchmark_results = [date_result, borough_result]
storage_results = [
    {**result, **get_storage_metrics(result["gold_target"])}
    for result in benchmark_results
]

print(json.dumps(storage_results, indent=2))

[
  {
    "design": "by_date",
    "partition_column": "date",
    "ingestion_seconds": 93.827,
    "gold_target": "s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_date/gold/gold_integrated_taxi_trips",
    "data_files": 165,
    "total_files": 167,
    "data_size_mib": 281.78,
    "total_size_mib": 282.23
  },
  {
    "design": "by_pickup_borough",
    "partition_column": "pickup_borough",
    "ingestion_seconds": 113.293,
    "gold_target": "s3a://delta-data/benchmark/gold_taxi/20260913T081248Z/by_pickup_borough/gold/gold_integrated_taxi_trips",
    "data_files": 32,
    "total_files": 34,
    "data_size_mib": 327.96,
    "total_size_mib": 328.14
  }
]


In [17]:
QUERY_NAMES = [
    "trips_per_borough",
    "average_trip_duration_per_day",
    "average_fare_per_borough",
]

def build_query(df, query_name):
    if query_name == "trips_per_borough":
        return (
            df.groupBy("pickup_borough")
            .agg(F.count("*").alias("trip_count"))
            .orderBy("pickup_borough")
        )

    if query_name == "average_trip_duration_per_day":
        duration_minutes = (
            F.unix_timestamp("dropoff_datetime")
            - F.unix_timestamp("pickup_datetime")
        ) / F.lit(60.0)
        return (
            df.groupBy("date")
            .agg(F.avg(duration_minutes).alias("average_duration_minutes"))
            .orderBy("date")
        )

    if query_name == "average_fare_per_borough":
        return (
            df.groupBy("pickup_borough")
            .agg(F.avg("fare_amount").alias("average_fare"))
            .orderBy("pickup_borough")
        )

    raise ValueError(f"Unknown query: {query_name}")

def measure_query_latency(target, query_name, repetitions=3):
    build_query(
        spark.read.format("delta").load(target), query_name
    ).collect()

    samples = []
    result_row_count = None
    for _ in range(repetitions):
        df = spark.read.format("delta").load(target)
        started_at = perf_counter()
        rows = build_query(df, query_name).collect()
        samples.append(perf_counter() - started_at)
        result_row_count = len(rows)

    return {
        "query": query_name,
        "median_latency_seconds": round(median(samples), 3),
        "samples_seconds": [round(sample, 3) for sample in samples],
        "result_row_count": result_row_count,
    }

In [18]:
query_results = []
for result in benchmark_results:
    for query_name in QUERY_NAMES:
        query_result = measure_query_latency(
            result["gold_target"], query_name
        )
        query_results.append({
            "design": result["design"],
            "partition_column": result["partition_column"],
            **query_result,
        })

print(json.dumps(query_results, indent=2))

[
  {
    "design": "by_date",
    "partition_column": "date",
    "query": "trips_per_borough",
    "median_latency_seconds": 0.366,
    "samples_seconds": [
      0.405,
      0.366,
      0.331
    ],
    "result_row_count": 8
  },
  {
    "design": "by_date",
    "partition_column": "date",
    "query": "average_trip_duration_per_day",
    "median_latency_seconds": 0.393,
    "samples_seconds": [
      0.411,
      0.393,
      0.381
    ],
    "result_row_count": 95
  },
  {
    "design": "by_date",
    "partition_column": "date",
    "query": "average_fare_per_borough",
    "median_latency_seconds": 0.341,
    "samples_seconds": [
      0.346,
      0.3,
      0.341
    ],
    "result_row_count": 8
  },
  {
    "design": "by_pickup_borough",
    "partition_column": "pickup_borough",
    "query": "trips_per_borough",
    "median_latency_seconds": 0.168,
    "samples_seconds": [
      0.168,
      0.149,
      0.171
    ],
    "result_row_count": 8
  },
  {
    "design": "by_pickup

### Print query output

In [21]:
display_value_columns = {
    "average_trip_duration_per_day": "average_duration_minutes",
    "average_fare_per_borough": "average_fare",
}

for result in benchmark_results:
    print(f"\n=== {result['design']} ({result['partition_column']}) ===")

    for query_name in QUERY_NAMES:
        print(f"\n--- {query_name} ---")

        df = spark.read.format("delta").load(result["gold_target"])
        query_df = build_query(df, query_name)

        value_column = display_value_columns.get(query_name)
        if value_column:
            query_df = query_df.withColumn(
                value_column,
                F.format_number(F.col(value_column), 2),
            )

        query_df.show(200, truncate=False)


=== by_date (date) ===

--- trips_per_borough ---
+--------------+----------+
|pickup_borough|trip_count|
+--------------+----------+
|Bronx         |24730     |
|Brooklyn      |95775     |
|EWR           |915       |
|Manhattan     |8442873   |
|N/A           |4916      |
|Queens        |817100    |
|Staten Island |220       |
|Unknown       |31304     |
+--------------+----------+


--- average_trip_duration_per_day ---
+----------+------------------------+
|date      |average_duration_minutes|
+----------+------------------------+
|2003-01-01|327.12                  |
|2009-01-01|35.48                   |
|2009-01-02|17.00                   |
|2024-01-01|16.40                   |
|2024-01-02|17.18                   |
|2024-01-03|16.69                   |
|2024-01-04|16.43                   |
|2024-01-05|15.25                   |
|2024-01-06|14.74                   |
|2024-01-07|13.87                   |
|2024-01-08|16.05                   |
|2024-01-09|15.31                   |
|20

### Export query output on csv files

In [22]:
value_columns = {
    "average_trip_duration_per_day": "average_duration_minutes",
    "average_fare_per_borough": "average_fare",
}

from pathlib import Path
import csv

RESULTS_DIRECTORY = Path("/opt/spark/notebooks/benchmark")
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

for result in benchmark_results:
    for query_name in QUERY_NAMES:
        df = spark.read.format("delta").load(result["gold_target"])
        query_df = build_query(df, query_name)

        value_column = value_columns.get(query_name)
        if value_column:
            query_df = query_df.withColumn(
                value_column,
                F.round(F.col(value_column), 2),
            )

        output_path = RESULTS_DIRECTORY / (
            f"{result['design']}_{query_name}.csv"
        )

        rows = query_df.collect()

        with output_path.open("w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(query_df.columns)

            for row in rows:
                writer.writerow([row[column] for column in query_df.columns])

        print(f"Wrote {output_path}")

Wrote /opt/spark/notebooks/benchmark/by_date_trips_per_borough.csv
Wrote /opt/spark/notebooks/benchmark/by_date_average_trip_duration_per_day.csv
Wrote /opt/spark/notebooks/benchmark/by_date_average_fare_per_borough.csv
Wrote /opt/spark/notebooks/benchmark/by_pickup_borough_trips_per_borough.csv
Wrote /opt/spark/notebooks/benchmark/by_pickup_borough_average_trip_duration_per_day.csv
Wrote /opt/spark/notebooks/benchmark/by_pickup_borough_average_fare_per_borough.csv


## Cleanup

Run the next cell only after all benchmark cells have completed.

In [ ]:
spark.stop()